In [ ]:
!pip install -q datasets transformers accelerate evaluate scikit-learn

In [ ]:
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

raw_datasets = load_dataset("ailsntua/QEvasion")

model_name = "answerdotai/ModernBERT-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_df = raw_datasets['train'].to_pandas()

train_df_split, val_df_split = train_test_split(
    train_df,
    test_size=700,
    random_state=42,
    stratify=train_df['evasion_label']
)

raw_datasets['train'] = Dataset.from_pandas(train_df_split, preserve_index=False)
raw_datasets['validation'] = Dataset.from_pandas(val_df_split, preserve_index=False)

label2id = {
    "Explicit": 0,
    "Implicit": 1,
    "Dodging": 2,
    "General": 3,
    "Deflection": 4,
    "Partial/half-answer": 5,
    "Declining to answer": 6,
    "Claims ignorance": 7,
    "Clarification": 8
}

id2label = {v: k for k, v in label2id.items()}

print(raw_datasets)

In [ ]:
from transformers import DataCollatorWithPadding

def tokenize_function(examples):
    inputs = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples["question"], examples["interview_answer"])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        max_length=3000,
        padding=False
    )
    tokenized["labels"] = [label2id[l] for l in examples["evasion_label"]]
    return tokenized

tokenized_datasets = {}
tokenized_datasets['train'] = raw_datasets['train'].map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets['train'].column_names
)
tokenized_datasets['validation'] = raw_datasets['validation'].map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets['validation'].column_names
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=9,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')

    return {
        'accuracy': accuracy,
        'f1': f1
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_modernbert_evasion",

    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=4,
    gradient_checkpointing=True,

    learning_rate=2e-5,
    weight_decay=0.1,
    warmup_ratio=0.06,
    num_train_epochs=5,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_dir="./logs",
    logging_steps=10,
    logging_strategy="steps",
    seed=42,

    fp16=True,
    bf16=False,
    optim="adamw_torch_fused",
    report_to="none",
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

print("Fine tuning complete.")

In [ ]:
from datasets import load_dataset
from sklearn.metrics import precision_recall_fscore_support, f1_score
import pandas as pd

test_dataset_raw = load_dataset("ailsntua/QEvasion")['test']

def tokenize_test(examples):
    texts = [
        f"Question: {q}\nAnswer: {a}"
        for q, a in zip(examples['question'], examples['interview_answer'])
    ]
    return tokenizer(texts, truncation=True, padding=False, max_length=3000)

test_tokenized = test_dataset_raw.map(
    tokenize_test,
    batched=True,
    remove_columns=[col for col in test_dataset_raw.column_names
                    if col not in ['annotator1', 'annotator2', 'annotator3', 'clarity_label']]
)

predictions_logits = trainer.predict(test_tokenized)
predictions = np.argmax(predictions_logits.predictions, axis=-1)

predicted_labels = [id2label[pred] for pred in predictions]

correct = 0
total = len(test_dataset_raw)

for i in range(total):
    pred = predicted_labels[i]
    annotator_labels = [
        test_dataset_raw[i]['annotator1'],
        test_dataset_raw[i]['annotator2'],
        test_dataset_raw[i]['annotator3']
    ]

    if pred in annotator_labels:
        correct += 1

accuracy_evasion = correct / total

all_labels = list(label2id.keys())
label_to_idx = {lbl: i for i, lbl in enumerate(all_labels)}

n = len(test_dataset_raw)
y_true_binary = np.zeros((n, len(all_labels)), dtype=int)
y_pred_binary = np.zeros((n, len(all_labels)), dtype=int)

for i in range(n):
    ann_set = {
        test_dataset_raw[i]['annotator1'],
        test_dataset_raw[i]['annotator2'],
        test_dataset_raw[i]['annotator3']
    }

    for lbl in ann_set:
        if lbl in label_to_idx:
            y_true_binary[i, label_to_idx[lbl]] = 1

    pred = predicted_labels[i]
    if pred in label_to_idx:
        y_pred_binary[i, label_to_idx[pred]] = 1

precisions, recalls, f1s, supports = precision_recall_fscore_support(
    y_true_binary, y_pred_binary,
    average=None,
    labels=range(len(all_labels)),
    zero_division=0
)

macro_f1_evasion = np.mean(f1s)
micro_f1_evasion = f1_score(y_true_binary, y_pred_binary, average='micro', zero_division=0)

y_pred_evasion = predicted_labels

print(f"Accuracy: {accuracy_evasion:.4f}")
print(f"Macro F1: {macro_f1_evasion:.4f}")
print(f"Micro F1: {micro_f1_evasion:.4f}")

print("\nPer-class metrics:")
for i, lbl in enumerate(all_labels):
    print(f"   {lbl:<25s} P={precisions[i]:.2f} R={recalls[i]:.2f} F1={f1s[i]:.2f} Support={supports[i]:d}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

evasion_to_clarity = {
    "Explicit": "Clear Reply",
    "Implicit": "Ambivalent",
    "Dodging": "Ambivalent",
    "General": "Ambivalent",
    "Deflection": "Ambivalent",
    "Partial/half-answer": "Ambivalent",
    "Declining to answer": "Clear Non-Reply",
    "Claims ignorance": "Clear Non-Reply",
    "Clarification": "Clear Non-Reply",
}

y_pred_clarity_hierarchical = [evasion_to_clarity[label] for label in y_pred_evasion]

y_true_clarity = [test_dataset_raw[i]['clarity_label'] for i in range(len(test_dataset_raw))]

accuracy_clarity_hierarchical = accuracy_score(y_true_clarity, y_pred_clarity_hierarchical)
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]
macro_f1_clarity_hierarchical = f1_score(
    y_true_clarity,
    y_pred_clarity_hierarchical,
    average='macro',
    labels=clarity_labels,
    zero_division=0
)


print(f"Accuracy: {accuracy_clarity_hierarchical:.4f}")
print(f"Macro F1: {macro_f1_clarity_hierarchical:.4f}")

print("\nPer-class metrics:")
print(classification_report(
    y_true_clarity,
    y_pred_clarity_hierarchical,
    labels=clarity_labels,
    zero_division=0
))

print("\nConfusion Matrix:")
cm_clarity = confusion_matrix(y_true_clarity, y_pred_clarity_hierarchical, labels=clarity_labels)
cm_clarity_df = pd.DataFrame(cm_clarity, index=clarity_labels, columns=clarity_labels)
print(cm_clarity_df)
